In [4]:
import re
import sqlite3
import pandas as pd

DB_PATH = "bosch_dw.db"
MIN_RATING = 4

In [5]:
CLAIMS = {
    "home_connect": [
        r"home\s*connect", r"\bapp\b", r"wi-?fi", r"smartphone",
        r"remote(ly)?", r"connected?\s+appliance",
    ],
    "smart_start": [
        r"smart\s*start", r"off.?peak", r"energy\s+tari",
        r"cheap(er)?\s+(rate|energy|electricity)", r"electricity\s+price",
    ],
    "programme_download": [
        r"programme\s+download", r"download.*program",
        r"new\s+program", r"update.*program",
    ],
    "intelligent_prog": [
        r"intelligent\s+programme", r"auto.?sens", r"sensor.*wash",
        r"auto\s+program", r"automatic(ally)?\s+(detect|select|adjust)",
    ],
    "wash_assistant": [
        r"wash\s+assistant", r"washing\s+assistant",
    ],
    "efficient_dry": [
        r"efficient\s*dry", r"dry(ing)?\s+result",
        r"dishes?\s+(are\s+|came?\s+out\s+)?dry", r"condensation\s+dry",
        r"automatic\s+door(\s+opening)?", r"door\s+(opening|open(s)?)",
        r"auto[\s-]?open",
        r"(door\s+)?opens?\s+(automatically|on\s+its\s+own|by\s+itself)",
        r"self[\s-]?opening(\s+door)?",
    ],
    "perfect_dry": [
        r"perfect\s*dry", r"plastic(s)?\s+(are\s+|came?\s+out\s+)?dry",
        r"dry(ing)?(\s+on)?\s+plastic", r"zeolite",
    ],
    "extra_clean_zone": [
        r"extra\s+clean\s+zone", r"intensive\s+zone", r"pots?\s+and\s+pans?",
        r"heavily\s+soil", r"intensiv",
    ],
    "hygiene_certified": [
        r"hygie(ne|nic)", r"bacteria", r"germ", r"sanitiz",
        r"disinfect", r"70.?°?.?c", r"certified", r"allergen",
    ],
    "remote_diagnostics": [
        r"remote\s+diagnos", r"error\s+(code|message)",
        r"technician", r"service\s+call", r"fault\s+detect",
    ],
    "green_collection": [
        r"green\s+collection", r"eco.?friendly", r"sustainab",
        r"recycled?\s+steel", r"environmental", r"carbon",
    ],
    "silent_power_drive": [
        r"silent\s*power\s*drive", r"\bsilent\b", r"\bquiet\b",
        r"\bnoise\b", r"\bnoisy\b", r"\bdB\b", r"decibel", r"sound\s+level",
    ],
    "aquastop": [
        r"aqua\s*stop", r"flood(ing)?", r"leak(age|s|ed|ing)?",
        r"water\s+(protection|damage|leak)", r"anti.?flood",
    ],
    "time_light": [
        r"time\s+light", r"remaining\s+time", r"countdown",
        r"floor\s+(light|project)", r"projected?\s+(on\s+)?floor", r"info\s*light",
    ],
    "emotion_light": [
        r"emotion\s+light", r"interior\s+light", r"inside\s+light",
        r"illuminat", r"light\s+inside",
    ],
    "open_assist": [
        r"open\s+assist", r"auto.?open", r"door\s+(open|pop)",
        r"opens?\s+automatically", r"automatic\s+door",
    ],
    "rackmatic": [
        r"rackmatic", r"adjustable\s+(rack|basket|height)",
        r"rack\s+height", r"height\s+adjust", r"upper\s+basket\s+adjust",
    ],
    "max_flex_pro": [
        r"max\s+flex\s+pro", r"flex(ible)?\s+(rack|basket|tine)",
        r"fold(able|ing)?\s+(tine|spike|prong)", r"large\s+(pot|pan|item)",
    ],
    "dosage_assist": [
        r"dosage\s+assist", r"detergent\s+dissolv", r"tablet\s+dissolv",
        r"dissolv.*detergent", r"detergent\s+compart",
    ],
    
    "vario_hinge": [
        r"vario\s*hinge", r"\bhinge\b", r"door\s+hinge",
        r"fold(ing)?\s+door", r"door\s+fold",
    ],
    "favorite_program": [
        r"favou?rite\s+program", r"saved\s+program",
        r"my\s+program", r"custom\s+program", r"personal\s+program",
        r"preferred\s+program",
    ],
}

CLAIM_EXACT_NAME = {
    "home_connect":       r"home\s*connect",
    "smart_start":        r"smart\s*start",
    "programme_download": r"programme\s*download",
    "intelligent_prog":   r"intelligent\s*programme",
    "wash_assistant":     r"wash\s*assistant",
    "efficient_dry":      r"efficient\s*dry",
    "perfect_dry":        r"perfect\s*dry",
    "extra_clean_zone":   r"extra\s*clean\s*zone",
    "hygiene_certified":  r"hygie\w+\s*certif\w+",
    "remote_diagnostics": r"remote\s*diagnos\w+",
    "green_collection":   r"green\s*collection",
    "silent_power_drive": r"silent\s*power\s*drive",
    "aquastop":           r"aqua\s*stop",
    "time_light":         r"time\s*light",
    "emotion_light":      r"emotion\s*light",
    "open_assist":        r"open\s*assist",
    "rackmatic":          r"rackmatic",
    "max_flex_pro":       r"max\s*flex\s*pro",
    "dosage_assist":      r"dosage\s*assist",
    "vario_hinge":        r"vario\s*hinge",
    "favorite_program":   r"favou?rite\s*program",
}

COMPILED = {
    claim: [re.compile(kw, re.IGNORECASE) for kw in patterns]
    for claim, patterns in CLAIMS.items()
}
COMPILED_EXACT = {
    claim: re.compile(pattern, re.IGNORECASE)
    for claim, pattern in CLAIM_EXACT_NAME.items()
}

In [6]:
with sqlite3.connect(DB_PATH) as conn:
    products = pd.read_sql("SELECT * FROM product_db", conn)
    reviews  = pd.read_sql("SELECT * FROM review_db",  conn)

print(f"Prodotti: {len(products)}")
print(f"Recensioni: {len(reviews)}")

Prodotti: 141
Recensioni: 15271


In [7]:
def matches_any(text: str, patterns: list) -> bool:
    return bool(text) and any(p.search(text) for p in patterns)

text_col = (reviews["title_en"].fillna("") + " " + reviews["text_en"].fillna(""))

for claim, patterns in COMPILED.items():
    reviews[f"mentions_{claim}"] = text_col.apply(lambda t: matches_any(t, patterns))

for claim, pattern in COMPILED_EXACT.items():
    reviews[f"exact_{claim}"] = text_col.apply(lambda t: bool(pattern.search(t)))

In [8]:
has_cols = [c for c in products.columns if c.startswith("has_")]

reviews = reviews.merge(
    products[["product_id"] + has_cols],
    on="product_id",
    how="left",
)

reviews[["product_id", "rating"] + has_cols].head(3)

,product_id,rating,has_aquastop,has_dosage_assist,has_efficient_dry,has_emotion_light,has_extra_clean_zone,has_favorite_program,has_green_collection,has_home_connect,...,has_perfect_dry,has_programme_download,has_rackmatic,has_remote_diagnostics,has_silent_power_drive,has_smart_start,has_status_light,has_time_light,has_vario_hinge,has_wash_assistant
0,SBH4ECX21E,5,1,1,1,0,0,1,0,1,...,0,1,1,1,1,0,1,0,1,1
1,SBH4ECX21E,4,1,1,1,0,0,1,0,1,...,0,1,1,1,1,0,1,0,1,1
2,SBH4ECX21E,5,1,1,1,0,0,1,0,1,...,0,1,1,1,1,0,1,0,1,1


In [9]:
filtered = reviews[reviews["rating"] >= MIN_RATING].copy()

rows = []
for claim in CLAIMS:
    has_col     = f"has_{claim}"
    mention_col = f"mentions_{claim}"

    if has_col not in filtered.columns:
        rows.append({"claim": claim, "eligible": None, "mentions": None, "mention_%": None})
        continue

    eligible = filtered[filtered[has_col] == 1]
    n        = len(eligible)
    mentions = eligible[mention_col].sum()
    exact    = eligible[f"exact_{claim}"].sum() if f"exact_{claim}" in eligible.columns else 0

    rows.append({
        "claim":     claim,
        "eligible":  n,
        "mentions":  int(mentions),
        "mention_%": round(100 * mentions / n, 1) if n else 0,
        "exact":     int(exact),
        "exact_%":   round(100 * exact / n, 1) if n else 0,
    })

stats = pd.DataFrame(rows).sort_values("mention_%", ascending=False)
stats

,claim,eligible,mentions,mention_%,exact,exact_%
11,silent_power_drive,14146,5553,39.3,0,0.0
0,home_connect,13296,1404,10.6,452,3.4
15,open_assist,304,27,8.9,4,1.3
5,efficient_dry,8782,609,6.9,85,1.0
6,perfect_dry,3385,181,5.3,74,2.2
13,time_light,2762,85,3.1,51,1.8
14,emotion_light,2640,66,2.5,1,0.0
16,rackmatic,679,14,2.1,0,0.0
7,extra_clean_zone,3271,66,2.0,9,0.3
17,max_flex_pro,1195,21,1.8,5,0.4


In [10]:
CLAIM_TO_INSPECT = "efficient_dry"

EXACT_ONLY = False

match_col = f"exact_{CLAIM_TO_INSPECT}" if EXACT_ONLY else f"mentions_{CLAIM_TO_INSPECT}"

mask = (
    (reviews["rating"] >= MIN_RATING) &
    (reviews[f"has_{CLAIM_TO_INSPECT}"] == 1) &
    (reviews[match_col])
)

sample = reviews[mask][["product_id", "country", "rating", "title_en", "text_en"]].head(10)
pd.set_option("display.max_colwidth", 120)
sample

,product_id,country,rating,title_en,text_en
1,SBH4ECX21E,DE,4,Good dishwasher with no frills,The dishwasher does its job cleanly and reliably. Integrating it into the Home Connect app is also as easy as usual....
8,SBH4ECX21E,BE,4,"Practical, fast and large dishwasher","A very practical dishwasher thanks to the dynamic layout of the baskets. The cutlery drawer is fine, but less useful..."
14,SBH4ECX21E,DE,4,Good dishwasher,"I looked for a model with a Vario hinge for a long time and decided on this model. I'm very satisfied, but I would h..."
21,SBH4ECX21E,BE,5,Great device,"I am satisfied with this new dishwasher from Bosch. Good washing results, dishes are dry. I find the cutlery drawer ..."
67,SBH4ECX21E,DE,5,Bosch fully integrated dishwasher,It's very good that the door opens automatically after you've finished rinsing and that you can set it to be quiet.
121,SBH4ECX28E,DE,4,Very good dishwasher,I have been using the product for a few weeks and am very satisfied so far. The Home Connect connection is also prac...
191,SMD6ECX12E,PL,5,This is what I was looking for,#opinionmotivated by promotion I really like Bosch equipment and I use many devices. This is my next Bosch dishwashe...
199,SMD6ECX12E,PL,5,Perfect. I recommend.,"#opinionmotivated by promotion* This dishwasher is a hit! Quiet, roomy and very convenient to use. What I like most ..."
212,SMD6ECX12E,PL,5,#opinionmotivated by promotion,"I highly recommend this dishwasher, it has: - great washing and drying efficiency - automatic door opening after the..."
213,SMD6ECX12E,PL,5,The BOSCH SMD6ECX12E dishwasher is a good choice,The BOSCH SMD6ECX12E dishwasher is a good choice. It works quietly and the dishes are washed after each cycle. I ap...
